In [32]:
import os, sys
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("../data/raw/wdi_solow_raw.csv")
df.head()

# Step 1: Melt wide → long
df_long = df.melt(
    id_vars=["country", "series"],
    var_name="year",
    value_name="value"
)

# Clean year format: YR1990 → 1990
df_long["year"] = df_long["year"].str.replace("YR", "").astype(int)

df_long.head()

indicator_map = {
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "NE.GDI.TOTL.ZS": "investment_rate",
    "SP.POP.GROW": "population_growth",
}

df_long["indicator"] = df_long["series"].map(indicator_map)

df_clean = df_long.pivot_table(
    index=["country", "year"],
    columns="indicator",
    values="value"
).reset_index()

df_clean.columns.name = None

df_clean.head()

df = df_clean.copy()

df = df[["country", "year", "gdp_per_capita", "investment_rate", "population_growth"]]

df = df.dropna(subset=["gdp_per_capita", "investment_rate", "population_growth"])

df["year"] = df["year"].astype(int)

g = 0.02
delta = 0.03

df["n_g_delta"] = df["population_growth"] / 100 + g + delta

df = df[
    (df["investment_rate"] > 0) &
    (df["gdp_per_capita"] > 0) &
    (df["n_g_delta"] > 0)
]

df["ln_y"] = np.log(df["gdp_per_capita"])
df["ln_s"] = np.log(df["investment_rate"] / 100)
df["ln_ngd"] = np.log(df["n_g_delta"])

df.head()
df_clean.to_csv("../data/processed/wdi_solow_tidy.csv", index=False)
df.to_csv("../data/processed/solow_panel.csv", index=False)
print("Saved to ../data/processed/solow_panel.csv")

X = df[["ln_s", "ln_ngd"]]
X = sm.add_constant(X)

y = df["ln_y"]

model = sm.OLS(y, X).fit()
print(model.summary())

# =========================
# Extended Solow Model
# =========================

df = pd.read_csv("../data/raw/wdi_solow_raw_extended.csv")
df.head()

df_long = df.melt(
    id_vars=["country", "series"],
    var_name="year",
    value_name="value"
)

df_long["year"] = df_long["year"].str.replace("YR", "", regex=False).astype(int)

indicator_map = {
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "NE.GDI.TOTL.ZS": "investment_rate",
    "SP.POP.GROW": "population_growth",
    "SE.SEC.ENRR": "school_enrollment"
}

df_long["indicator"] = df_long["series"].map(indicator_map)

df_clean = df_long.pivot_table(
    index=["country", "year"],
    columns="indicator",
    values="value"
).reset_index()

df_clean.columns.name = None
df_clean.head()

df = df_clean.copy()

df = df[[
    "country",
    "year",
    "gdp_per_capita",
    "investment_rate",
    "population_growth",
    "school_enrollment"
]]

df = df.dropna(subset=[
    "gdp_per_capita",
    "investment_rate",
    "population_growth",
    "school_enrollment"
])

df["year"] = df["year"].astype(int)

g = 0.02
delta = 0.03

df["n_g_delta"] = df["population_growth"] / 100 + g + delta

df = df[
    (df["gdp_per_capita"] > 0) &
    (df["investment_rate"] > 0) &
    (df["n_g_delta"] > 0) &
    (df["school_enrollment"] > 0)
].copy()

df["ln_y"] = np.log(df["gdp_per_capita"])
df["ln_s"] = np.log(df["investment_rate"] / 100)
df["ln_ngd"] = np.log(df["n_g_delta"])
df["ln_h"] = np.log(df["school_enrollment"])

df.head()

X = df[["ln_s", "ln_ngd", "ln_h"]]
X = sm.add_constant(X)

y = df["ln_y"]

model_aug = sm.OLS(y, X).fit()
print(model_aug.summary())

df[["ln_s", "ln_h", "ln_ngd"]].corr()

from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[["ln_s", "ln_ngd", "ln_h"]]
X = sm.add_constant(X)

vif = pd.DataFrame()
vif["variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif)

df[["ln_s", "ln_h"]].describe()

Saved to ../data/processed/solow_panel.csv
                            OLS Regression Results                            
Dep. Variable:                   ln_y   R-squared:                       0.195
Model:                            OLS   Adj. R-squared:                  0.193
Method:                 Least Squares   F-statistic:                     95.35
Date:                Sat, 28 Mar 2026   Prob (F-statistic):           8.52e-38
Time:                        10:38:46   Log-Likelihood:                -1298.4
No. Observations:                 788   AIC:                             2603.
Df Residuals:                     785   BIC:                             2617.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const    